# Differentiable optimization: the closure core

The heart of the [closure / reconstruction methods](../reco): a differentiable
forward, optimized to make the simulated readout match an observed event. Here,
stripped to essentials — reconstruct an event as **N point charges** `[x,y,z,dE]`
with the **Sobolev loss** and Adam. The reco notebooks add MCMC relocation and
real-scale events.

Key closure detail kept here: the energy `dE` is optimized with a **much smaller**
learning rate (`lr_e_mult`) and floored at `min_energy`, so it converges steadily
instead of collapsing. Self-contained (synthetic truth; no external data).


In [ ]:
# Resolve the repo root so imports and config/ paths work from any folder.
import os, sys
_d = os.path.abspath(os.getcwd())
while _d != os.path.dirname(_d) and not os.path.isdir(os.path.join(_d, 'config')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)


In [ ]:
import numpy as np, jax, jax.numpy as jnp, optax, matplotlib.pyplot as plt
from tools.simulation import DetectorSimulator
from tools.geometry import generate_detector
from tools.loader import build_deposit_data
from tools.losses import sobolev_loss_geomean_log1p, make_sobolev_weight
detector = generate_detector('config/cubic_wireplane_config.yaml')
N = 600; DX_MM = 4.0
PLANE_NAMES = ['U0','V0','Y0','U1','V1','Y1']


## 1. Truth event → target signals


In [ ]:
rng = np.random.RandomState(0)
truth_pos = np.stack([np.linspace(-180, -30, N),
                      np.linspace(-60, 60, N),
                      np.linspace(-40, 70, N)], 1).astype(np.float32) * 10
truth_de = np.full(N, 2.2 * DX_MM/10, np.float32)
# modified_box recombination (the closure default); same model for truth and forward
sim_truth = DetectorSimulator(detector, use_bucketed=False, total_pad=10000,
    response_chunk_size=10000, include_track_hits=False, recombination_model='modified_box')
cfg = sim_truth.config
deposits = build_deposit_data(truth_pos, truth_de, np.full(N, DX_MM/10, np.float32),
                              cfg, track_ids=np.zeros(N, np.int32))
resp, _, _ = sim_truth.process_event(deposits)
truth = [jnp.zeros((1,1))]*6; weights = [jnp.zeros((1,1))]*6; active = []
for (v,p), s in resp.items():
    s = jnp.asarray(s); idx = v*3+p; truth[idx] = s
    if jnp.any(s != 0):
        active.append(idx); H,W = s.shape; weights[idx] = make_sobolev_weight(H, W, s=1.0)
active = sorted(active); truth = tuple(truth); weights = tuple(weights)
print('active planes:', [PLANE_NAMES[i] for i in active], '| truth total dE = %.1f MeV' % truth_de.sum())


## 2. Differentiable forward + Sobolev loss
Convolving with the Sobolev kernel `1/(k²+κ²)^{s/2}` gives non-overlapping signals a
usable gradient (plain MSE would be ~flat).


In [ ]:
sim_opt = DetectorSimulator(detector, differentiable=True, n_segments=N, total_pad=10000,
                            recombination_model='modified_box')
sp = sim_opt.default_sim_params
def forward(params):
    return sim_opt.forward_segments(sp, params[:, :3], params[:, 3], dx=DX_MM)
def loss_fn(params):
    return sobolev_loss_geomean_log1p(forward(params), truth, weights, planes=tuple(active))
grad_fn = jax.jit(jax.value_and_grad(loss_fn))  # first call compiles (~1 min)


## 3. Initialize: jittered truth point charges (50 mm, ±30% energy)


In [ ]:
init = np.concatenate([
    truth_pos + rng.normal(0, 50, (N, 3)).astype(np.float32),
    (truth_de * rng.uniform(0.7, 1.3, N)).reshape(-1, 1)], 1)
params = jnp.asarray(init)
L, g = grad_fn(params)
print(f'initial loss = {float(L):.4f}   grad shape {g.shape}')
print(f'  |dL/d(x,y,z)| mean = {float(jnp.mean(jnp.abs(g[:,:3]))):.3e}'
      f'   |dL/d(dE)| mean = {float(jnp.mean(jnp.abs(g[:,3]))):.3e}')


## 4. Optimize (Adam, with the closure's per-parameter learning rates)


In [ ]:
LR_E_MULT, MIN_E = 0.01, 0.001     # energy moves ~100x slower, floored (closure recipe)
opt = optax.adam(optax.exponential_decay(1.0, 1, 0.999), b1=0.9, b2=0.999)
st = opt.init(params)
losses = []
for i in range(80):
    L, g = grad_fn(params)
    upd, st = opt.update(g, st, params)
    upd = upd.at[:, 3].multiply(LR_E_MULT)
    params = optax.apply_updates(params, upd)
    params = params.at[:, 3].set(jnp.maximum(params[:, 3], MIN_E))
    losses.append(float(L))
    if i % 16 == 0 or i == 79:
        print(f'  step {i:2d}   loss {float(L):.4f}   total dE {float(jnp.sum(params[:,3])):.1f} MeV'
              f'  (truth {float(truth_de.sum()):.1f})')


## 5. Results: loss curve + truth vs reconstruction (Y plane)


In [ ]:
recon = forward(params); yp = 2
T = np.abs(np.asarray(truth[yp])); R = np.abs(np.asarray(recon[yp]))
vmax = np.percentile(T[T > 0], 99) if np.any(T > 0) else 1.0
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(losses); ax[0].set(title='Sobolev loss', xlabel='Adam step', ylabel='loss'); ax[0].grid(alpha=0.3)
for a, img, ttl in [(ax[1], T, 'truth (Y)'), (ax[2], R, 'reconstruction (Y)')]:
    a.imshow(img.T, aspect='auto', origin='lower', cmap='inferno', vmin=0, vmax=vmax)
    a.set(title=ttl, xlabel='wire', ylabel='time')
fig.suptitle(f'Minimal closure: loss {losses[0]:.3f} -> {losses[-1]:.3f}')
plt.tight_layout(); plt.show()


## Next
- `../reco/segments_closure.ipynb` — the full segments closure (adds MCMC relocation)
- `gradient_visualizations.ipynb`, `forward_segments.ipynb`
